In [ ]:
import json
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
import itertools

# Set the visual style for all plots
sns.set_theme(style="whitegrid", context="notebook", palette="Set1", font_scale=1.2)
# Use a colorblind-friendly palette for multiple libraries
PALETTES = {2: "Set1", 3: "Set1", 4: "Set2", 5: "Set2", 6: "tab10"}


def load_benchmark_data(filepath):
    """Load benchmark data from JSON file."""
    with open(filepath, "r") as f:
        return json.load(f)


def combine_datasets(datasets_dict):
    """
    Combine multiple datasets for comparison.

    Parameters:
    -----------
    datasets_dict : dict
        Dictionary with format {framework_name: dataset}

    Returns:
    --------
    list
        Combined list of benchmark entries with framework labels
    """
    all_data = []

    # Process each dataset with its framework name
    for framework_name, data in datasets_dict.items():
        for entry in data:
            # Add framework label
            entry["framework"] = framework_name

            # Convert time to milliseconds for better readability
            entry["mean_time_ms"] = entry["mean_time"] * 1000
            entry["min_time_ms"] = entry["min_time"] * 1000
            entry["max_time_ms"] = entry["max_time"] * 1000
            entry["median_time_ms"] = entry["median_time"] * 1000

            # Add configuration key for easier filtering
            entry["config"] = (
                f"sr={entry['sample_rate']}, ch={entry['channels']}, dur={entry['duration_sec']}s"
            )

            all_data.append(entry)

    return all_data


def create_dataframe(benchmark_data):
    """Convert benchmark data to a pandas DataFrame for analysis."""
    return pd.DataFrame(benchmark_data)


def plot_sample_type_comparison(
    df,
    operation,
    sample_rate,
    channels,
    duration,
    metric="mean_time_ms",
    output_dir=None,
):
    """
    Generate bar charts comparing performance across different sample types
    for a specific configuration.

    Parameters:
    -----------
    df : DataFrame
        The benchmark data
    operation : str
        'read' or 'write'
    sample_rate : int
        Sample rate in Hz
    channels : int
        Number of channels
    duration : int
        Duration in seconds
    metric : str
        The metric to plot (default: 'mean_time_ms')
    output_dir : str, optional
        Directory to save plots
    """
    # Filter data for the specific configuration
    filtered_data = df[
        (df["operation"] == operation)
        & (df["sample_rate"] == sample_rate)
        & (df["channels"] == channels)
        & (df["duration_sec"] == duration)
    ]

    if filtered_data.empty:
        print(
            f"No data for {operation}, sr={sample_rate}, ch={channels}, dur={duration}s"
        )
        return

    # Get number of frameworks for palette selection
    num_frameworks = len(filtered_data["framework"].unique())
    palette = PALETTES.get(num_frameworks, "tab10")

    # Set up plot title and ylabel based on metric
    if metric == "mean_time_ms":
        y_label = "Time (milliseconds ms)"
        title = f"{operation.capitalize()} Time by Sample Type\nsr={sample_rate}Hz, ch={channels}, dur={duration}s"
    elif metric == "throughput_bytes_per_sec":
        y_label = "Throughput (bytes/second)"
        title = f"{operation.capitalize()} Throughput by Sample Type\nsr={sample_rate}Hz, ch={channels}, dur={duration}s"

    # Create plot
    plt.figure(figsize=(14, 8))
    chart = sns.barplot(
        data=filtered_data,
        x="format",
        y=metric,
        hue="framework",
        palette=palette,
        edgecolor="black",
    )
    plt.title(title, fontsize=24)
    plt.xlabel("Sample Type", fontsize=20)
    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18)
    plt.ylabel(y_label, fontsize=20)

    plt.legend(
        title="Library",
        bbox_to_anchor=(0.5, -0.4),
        loc="lower center",
        ncols=num_frameworks,
        fontsize=16,
    )
    plt.tight_layout()

    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        filename = f"{operation}_{metric}_{sample_rate}Hz_{channels}ch_{duration}s.png"
        plt.savefig(f"{output_dir}/{filename}", dpi=300)
        plt.close()
    else:
        plt.show()


def generate_all_sample_type_plots(df, output_dir=None):
    """
    Generate sample type comparison plots for all configurations.

    Parameters:
    -----------
    df : DataFrame
        The benchmark data
    output_dir : str, optional
        Directory to save plots
    """
    if output_dir:
        Path(output_dir).mkdir(parents=True, exist_ok=True)

    # Get unique values for each parameter
    sample_rates = df["sample_rate"].unique()
    channels_list = df["channels"].unique()
    durations = df["duration_sec"].unique()
    operations = df["operation"].unique()
    metrics = ["mean_time_ms", "throughput_bytes_per_sec"]

    # Generate plots for all combinations
    for operation, sample_rate, channels, duration, metric in itertools.product(
        operations, sample_rates, channels_list, durations, metrics
    ):
        plot_sample_type_comparison(
            df, operation, sample_rate, channels, duration, metric, output_dir
        )


def generate_grid_of_plots(df, operation, metric="mean_time_ms", output_dir=None):
    """
    Generate a grid of sample type comparison plots for a specific operation.

    Parameters:
    -----------
    df : DataFrame
        The benchmark data
    operation : str
        'read' or 'write'
    metric : str
        The metric to plot (default: 'mean_time_ms')
    output_dir : str, optional
        Directory to save plots
    """
    # Get unique values for each parameter
    sample_rates = sorted(df["sample_rate"].unique())
    channels_list = sorted(df["channels"].unique())
    durations = sorted(df["duration_sec"].unique())

    # Get number of frameworks for palette selection
    num_frameworks = len(df["framework"].unique())
    palette = PALETTES.get(num_frameworks, "tab10")

    # Set up plot title and ylabel based on metric
    if metric == "mean_time_ms":
        y_label = "Time (milliseconds)"
        title_prefix = f"{operation.capitalize()} Time by Sample Type"
    elif metric == "throughput_bytes_per_sec":
        y_label = "Throughput (bytes/second)"
        title_prefix = f"{operation.capitalize()} Throughput by Sample Type"

    # Calculate grid dimensions
    n_configs = len(sample_rates) * len(channels_list) * len(durations)
    cols = min(3, n_configs)
    rows = (n_configs + cols - 1) // cols

    # Create figure
    fig, axes = plt.subplots(rows, cols, figsize=(10 * cols, 6 * rows), squeeze=False)

    # Counter for subplot position
    plot_idx = 0

    # Generate plots for all combinations
    for duration, sample_rate, channels in itertools.product(
        durations, sample_rates, channels_list
    ):
        # Calculate subplot position
        row = plot_idx // cols
        col = plot_idx % cols

        # Filter data for the specific configuration
        filtered_data = df[
            (df["operation"] == operation)
            & (df["sample_rate"] == sample_rate)
            & (df["channels"] == channels)
            & (df["duration_sec"] == duration)
        ]

        if not filtered_data.empty:
            # Create subplot
            ax = axes[row, col]
            sns.barplot(
                data=filtered_data,
                x="format",
                y=metric,
                hue="framework",
                palette=palette,
                ax=ax,
                edgecolor="black",
            )
            ax.set_title(
                f"sr={sample_rate}Hz, ch={channels}, dur={duration}s", fontsize=24
            )
            ax.set_xlabel("Sample Type", fontsize=20)
            ax.set_ylabel(y_label, fontsize=20)
            ax.tick_params(axis="x", labelsize=18)
            ax.tick_params(axis="y", labelsize=18)

            ax.get_legend().remove()

        plot_idx += 1

    # Remove any unused subplots
    for idx in range(plot_idx, rows * cols):
        row = idx // cols
        col = idx % cols
        fig.delaxes(axes[row, col])

    # Add overall title
    fig.suptitle(title_prefix, fontsize=28)

    ## create a legend underneath the plots
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=num_frameworks, fontsize=16)
    fig.tight_layout(
        rect=[0, 0.009, 1.0, 0.975]
    )  # Make room for the suptitle and legend

    if output_dir:
        filename = f"{operation}_{metric}_grid.png"
        plt.savefig(f"{output_dir}/{filename}", dpi=300)
        plt.close()
    else:
        plt.show()


def generate_summary_table(df, operation, metric="mean_time_ms"):
    """
    Generate a summary table of performance by sample type.

    Parameters:
    -----------
    df : DataFrame
        The benchmark data
    operation : str
        'read' or 'write'
    metric : str
        The metric to summarize (default: 'mean_time_ms')

    Returns:
    --------
    DataFrame
        Summary table
    """
    # Filter data for the specified operation
    op_data = df[df["operation"] == operation]

    # Create a pivot table
    pivot = pd.pivot_table(
        op_data,
        values=metric,
        index=["sample_rate", "channels", "duration_sec"],
        columns=["framework", "format"],
        aggfunc="mean",
    )

    # Format the index for better readability
    pivot.index = [f"sr={sr}Hz, ch={ch}, dur={dur}s" for sr, ch, dur in pivot.index]

    return pivot


def generate_markdown_table(
    df,
    operation,
    frameworks=None,
    metric="mean_time",
    channels=None,
    durations=None,
    sample_rates=None,
    include_throughput=False,
):
    """
    Generate a markdown table of benchmark results with times in microseconds.

    Parameters:
    -----------
    df : DataFrame
        The benchmark data
    operation : str
        'read' or 'write'
    frameworks : list, optional
        List of frameworks to include (default: None = all)
    metric : str
        The time metric to use (default: 'mean_time')
    channels : list, optional
        List of channel values to include (default: None = all)
    durations : list, optional
        List of duration values to include in seconds (default: None = all)
    sample_rates : list, optional
        List of sample rate values to include (default: None = all)
    include_throughput : bool
        Whether to include throughput in the table (default: False)

    Returns:
    --------
    str
        Markdown formatted table
    """
    # Filter by operation
    op_data = df[df["operation"] == operation].copy()

    # Apply additional filters if specified
    if frameworks is not None:
        op_data = op_data[op_data["framework"].isin(frameworks)]

    if channels is not None:
        op_data = op_data[op_data["channels"].isin(channels)]

    if durations is not None:
        op_data = op_data[op_data["duration_sec"].isin(durations)]

    if sample_rates is not None:
        op_data = op_data[op_data["sample_rate"].isin(sample_rates)]

    if op_data.empty:
        return "No data available for the specified filters."

    # Convert time to microseconds
    op_data[f"{metric}_us"] = op_data[metric] * 1_000_000

    # Create a pivot table with frameworks and formats as columns
    pivot = pd.pivot_table(
        op_data,
        values=f"{metric}_us",
        index=["channels", "duration_sec", "sample_rate"],
        columns=["framework", "format"],
        aggfunc="mean",
    )

    # Reset index to convert to a regular DataFrame with proper column order
    result_df = pivot.reset_index()

    # Rename columns for clarity
    result_df = result_df.rename(
        columns={
            "channels": "Channels",
            "duration_sec": "Duration (s)",
            "sample_rate": "Sample Rate (Hz)",
        }
    )

    # Format the values to 2 decimal places for better readability
    for col in result_df.columns[
        3:
    ]:  # Skip the first three columns which are the index
        result_df[col] = result_df[col].round(2)

    # Add a descriptive title
    title = (
        f"# {operation.capitalize()} Operation - Time in Microseconds"
        + "(\u03bcs) "
        + f"({metric.replace('_', ' ').title()})\n\n"
    )

    # Convert to markdown
    md_table = title + result_df.to_markdown(index=False)

    # If throughput is requested, create a separate table for it
    if include_throughput:
        throughput_pivot = pd.pivot_table(
            op_data,
            values="throughput_bytes_per_sec",
            index=["channels", "duration_sec", "sample_rate"],
            columns=["framework", "format"],
            aggfunc="mean",
        )

        # Reset index to convert to a regular DataFrame with proper column order
        throughput_df = throughput_pivot.reset_index()

        # Rename columns for clarity
        throughput_df = throughput_df.rename(
            columns={
                "channels": "Channels",
                "duration_sec": "Duration (s)",
                "sample_rate": "Sample Rate (Hz)",
            }
        )

        # Format the throughput values
        for col in throughput_df.columns[3:]:
            throughput_df[col] = throughput_df[col].round(0)

        # Add throughput table
        md_table += (
            f"\n\n# {operation.capitalize()} Operation - Throughput (Bytes/Second)\n\n"
        )
        md_table += throughput_df.to_markdown(index=False)

    return md_table


def analyze_benchmark_data(benchmark_paths, framework_names=None, output_dir=None):
    """
    Main function to analyze benchmark data from multiple sources.

    Parameters:
    -----------
    benchmark_paths : dict or list
        If dict: {framework_name: path_to_json}
        If list: List of paths to benchmark JSON files
    framework_names : list, optional
        List of names to use for each benchmark (required if benchmark_paths is a list)
    output_dir : str, optional
        Directory to save plots

    Returns:
    --------
    DataFrame
        The combined benchmark data
    """
    # Process input to get framework names and paths
    if isinstance(benchmark_paths, dict):
        # Dictionary format: {framework_name: path}
        datasets_dict = {
            name: load_benchmark_data(path) for name, path in benchmark_paths.items()
        }
    elif isinstance(benchmark_paths, list):
        # List format with separate framework names
        if framework_names is None or len(framework_names) != len(benchmark_paths):
            raise ValueError(
                "When benchmark_paths is a list, framework_names must be provided "
                + "with the same length as benchmark_paths"
            )
        datasets_dict = {
            name: load_benchmark_data(path)
            for name, path in zip(framework_names, benchmark_paths)
        }
    else:
        # Single path with default framework name
        datasets_dict = {"Benchmark": load_benchmark_data(benchmark_paths)}

    # Combine datasets
    all_data = combine_datasets(datasets_dict)

    # Create DataFrame
    df = create_dataframe(all_data)

    # Generate visualizations
    if output_dir:
        Path(output_dir).mkdir(parents=True, exist_ok=True)

    # Generate individual plots
    generate_all_sample_type_plots(df, output_dir)

    # Generate grid plots
    for operation in ["read", "write"]:
        generate_grid_of_plots(df, operation, "mean_time_ms", output_dir)
        generate_grid_of_plots(df, operation, "throughput_bytes_per_sec", output_dir)

    # Generate summary tables
    read_summary = generate_summary_table(df, "read", "mean_time_ms")
    write_summary = generate_summary_table(df, "write", "mean_time_ms")

    print("Read Performance Summary (milliseconds):")
    print(read_summary)
    print("\nWrite Performance Summary (milliseconds):")
    print(write_summary)

    print(
        f"Analysis complete. {'Plots saved to ' + output_dir if output_dir else 'Plots displayed.'}"
    )

    return df

In [ ]:
# Define paths to benchmark results
benchmark_paths = {
    "libsndfile (C)": "libsndfile_benchmark_results.json",
    "wavers (Rust)": "wavers_bench/wavers_benchmark_results.json",
    "hound (Rust)": "hound_bench/hound_benchmark_results.json",
}

output_directory = "benchmark_plots"

# Run analysis
df = analyze_benchmark_data(benchmark_paths, output_dir=output_directory)

# Create markdown table for a specific operation and configuration

read_tables = []
write_tables = []
for channels in [1, 2]:
    for duration in [1, 10]:
        for sample_rate in [8000, 16100, 44100]:
            read_table = generate_markdown_table(
                df,
                operation="read",
                metric="mean_time",
                channels=[channels],
                durations=[duration],
                sample_rates=[sample_rate],
                include_throughput=True,
            )
            read_tables.append(read_table)

            write_table = generate_markdown_table(
                df,
                operation="write",
                metric="mean_time",
                channels=[channels],
                durations=[duration],
                sample_rates=[sample_rate],
                include_throughput=True,
            )
            write_tables.append(write_table)

# Combine all tables into a single markdown string, interleaving read and write tables
combined_tables = []
for read_table, write_table in zip(read_tables, write_tables):
    combined_tables.append(read_table)
    combined_tables.append(write_table)

# Join all tables into a single markdown string
combined_markdown = "\n\n".join(combined_tables)

print(combined_markdown)

# Save the combined markdown to a file
with open("benchmark_results.md", "w") as f:
    f.write(combined_markdown)

Read Performance Summary (milliseconds):
framework                 hound (Rust)                       libsndfile (C)  \
format                        IEEE_F32     PCM_16     PCM_32       IEEE_F32   
sr=8000Hz, ch=1, dur=1s       0.028410   0.052323   0.034645       0.007498   
sr=8000Hz, ch=1, dur=10s      0.286950   0.513538   0.334481       0.013502   
sr=8000Hz, ch=1, dur=60s      1.727321   3.091043   1.987618       0.080338   
sr=8000Hz, ch=2, dur=1s       0.056260   0.104270   0.068340       0.008881   
sr=8000Hz, ch=2, dur=10s      0.558763   1.036972   0.673300       0.021210   
sr=8000Hz, ch=2, dur=60s      3.634131   6.151610   3.986614       0.147078   
sr=16000Hz, ch=1, dur=1s      0.056470   0.104205   0.068547       0.008352   
sr=16000Hz, ch=1, dur=10s     0.567322   1.036005   0.653273       0.020912   
sr=16000Hz, ch=1, dur=60s     3.572898   6.286236   4.102624       0.144944   
sr=16000Hz, ch=2, dur=1s      0.115226   0.207780   0.135280       0.010340   
sr=16000Hz,